In [ ]:
# Resolve to the repo root so relative paths (config.yaml, Data/, Resources/)
# work whether this notebook runs from the repo root or from notebooks/.
import os
while not os.path.exists('config.yaml'):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        raise RuntimeError('repo root (config.yaml) not found above cwd')
    os.chdir(parent)

# High-Paying Jobs Data Cleaning & Preparation
This notebook provides a professional, step-by-step workflow for cleaning and preparing U.S. high-paying jobs data. It integrates data from the Bureau of Labor Statistics (BLS) and the U.S. Census to enable analysis of salary, education, and occupation across states.

**Workflow Overview:**
- Load and clean BLS data (job and wage statistics)
- Load and clean Census data (demographics, education, occupation)
- Merge datasets for unified analysis
- Save cleaned outputs for downstream analytics

*All steps are clearly commented and follow best practices for reproducibility and clarity.*

# 2.  Data Preparation


In [ ]:
# Import required libraries for data processing
import pandas as pd

## BLS Data Preparation

In [ ]:
# Load BLS data (job and wage statistics) using openpyxl engine for .xlsx support
bls_data = pd.read_excel("./Resources/bls_state_data.xlsx", engine="openpyxl")
display(bls_data.head(3))  # Preview first rows to confirm successful load

In [ ]:
# Preview last rows to check for trailing issues or anomalies
display(bls_data.tail(3))

In [ ]:
# Check column names for consistency and expected structure
bls_data.columns

In [ ]:
# Select relevant columns for analysis
relevant_columns = [
    'AREA_TITLE', 'PRIM_STATE', 'LOC_QUOTIENT', 'OCC_CODE', 'OCC_TITLE',
    'TOT_EMP', 'JOBS_1000', 'H_MEAN', 'A_MEAN'
# ...add or remove columns as needed for your analysis
]
bls_data = bls_data[relevant_columns]
display(bls_data.head())  # Confirm selection

In [ ]:
# Check data types to ensure correct formats for analysis
bls_data.dtypes

In [ ]:
# Create a copy for cleaning and convert numeric columns
bls_df = bls_data.copy()
numeric_columns = ['LOC_QUOTIENT', 'TOT_EMP', 'H_MEAN', 'A_MEAN', 'JOBS_1000']
bls_df[numeric_columns] = bls_df[numeric_columns].apply(pd.to_numeric, errors='coerce')
display(bls_df.head())  # Preview cleaned data

In [ ]:
# Recheck data types after cleaning
bls_df.dtypes

In [ ]:
# Standardize string columns: remove spaces and set title case
bls_df['AREA_TITLE'] = bls_df['AREA_TITLE'].str.strip().str.title()
bls_df['OCC_TITLE'] = bls_df['OCC_TITLE'].str.strip().str.title()

In [ ]:
# Clean OCC_CODE: remove hyphens and drop invalid codes
bls_df['OCC_CODE'] = bls_df['OCC_CODE'].str.replace('-', '', regex=False)
bls_df = bls_df[bls_df['OCC_CODE'].notna() & (bls_df['OCC_CODE'] != '')]

In [ ]:
# (Optional) Check unique values for each column if needed for EDA
unique_values = bls_df.apply(lambda x: x.unique())

In [ ]:
# Check and count missing values in BLS data
missing_values = bls_df.isna().sum()
missing_values

In [ ]:
# Drop rows with any missing values for clean analysis
bls_df = bls_df.dropna(how='any')
bls_df.shape  # Check resulting shape

In [ ]:
# Keep only the 50 US states (exclude territories and DC)
unwanted_states = ['GU', 'PR', 'VI', 'DC']
bls_clean = bls_df[~bls_df['PRIM_STATE'].isin(unwanted_states)]
bls_clean['PRIM_STATE'].nunique()  # Should be 50

In [ ]:
# Filter for high-paying jobs: annual mean wage >= $100K or hourly mean >= $48.08
bls_clean = bls_clean[(bls_clean['A_MEAN'] >= 100000) | (bls_clean['H_MEAN'] >= 48.08)]
display(bls_clean.tail(5))

In [ ]:
# Check data types after filtering for high-paying jobs
bls_clean.dtypes

In [ ]:
# Summary statistics for numerical columns in cleaned BLS data
bls_clean.describe()

In [ ]:
# Check structure and info of cleaned BLS data
bls_clean.info()

In [ ]:
# Save cleaned BLS data to CSV for downstream analysis
bls_clean.to_csv('./Data/bls_data.csv', index=False)

## Census Data Preparation

In [ ]:
#load data
census_data = pd.read_csv("./Resources/census_data.csv",delimiter=',')
# check data
census_data.head()

In [ ]:
#Check the columns names
census_data.columns

In [ ]:
# Filter for individuals earning $100K or more
census_data= census_data[census_data['INCTOT'] >= 100000]

In [ ]:
# filtre the data  to only keep the relevant columns
relevant_c = ['STATEICP' ,'DEGFIELDD', 'EDUCD','OCCSOC','INCTOT','SEX', 'AGE']
census_data=census_data[relevant_c]
census_data.head()

In [ ]:
# Remove any non-numeric characters and ensure all codes are exactly 6 digits by padding with leading zeros
census_data.loc[:, 'OCCSOC'] = census_data['OCCSOC'].str.extract('(\d+)', expand=False).str.zfill(6)

In [ ]:
#Replace all occurrences of 1 with "Male" and 2 with "Female" in the SEX column.#
census_data.loc[:,'SEX'] = census_data['SEX'].astype(str).replace({'1': 'Male', '2': 'Female'})

In [ ]:
# Display the data Structure 
census_data.head()

In [ ]:
# Dictionary to map STATEICP codes to state names (50 U.S. states only)
census_df=census_data.copy()
state_map = {
    1: 'Connecticut', 2: 'Maine', 3: 'Massachusetts', 4: 'New Hampshire',
    5: 'Rhode Island', 6: 'Vermont', 11: 'Delaware', 12: 'New Jersey',
    13: 'New York', 14: 'Pennsylvania', 21: 'Illinois', 22: 'Indiana',
    23: 'Michigan', 24: 'Ohio', 25: 'Wisconsin', 31: 'Iowa', 32: 'Kansas',
    33: 'Minnesota', 34: 'Missouri', 35: 'Nebraska', 36: 'North Dakota',
    37: 'South Dakota', 40: 'Virginia', 41: 'Alabama', 42: 'Arkansas',
    43: 'Florida', 44: 'Georgia', 45: 'Louisiana', 46: 'Mississippi',
    47: 'North Carolina', 48: 'South Carolina', 49: 'Texas', 51: 'Kentucky',
    52: 'Maryland', 53: 'Oklahoma', 54: 'Tennessee', 56: 'West Virginia',
    61: 'Arizona', 62: 'Colorado', 63: 'Idaho', 64: 'Montana', 65: 'Nevada',
    66: 'New Mexico', 67: 'Utah', 68: 'Wyoming', 71: 'California',
    72: 'Oregon', 73: 'Washington', 81: 'Alaska', 82: 'Hawaii'
}
# apply the map on the Data
census_df.loc[:,'STATE']=census_df['STATEICP'].map(state_map)


In [ ]:
# Education map for the provided codes
education_map = {

    101: "Bachelor's degree", 
    114: "Master's degree", 
    115: "Professional degree", 
    116: "Doctoral degree", 
}

# Apply the mapping again
census_df['EDUCATION_LABEL'] = census_df['EDUCD'].map(education_map)

In [ ]:
degree_field_map = {
    1100: "Agriculture", 1101: "Agriculture Production and Management", 1102: "Agricultural Economics", 1103: "Animal Sciences", 1104: "Food Science",
    1105: "Plant Science", 1106: "Soil Science", 1199: "Miscellaneous Agriculture", 
    1301: "Environmental Science", 1302: "Forestry", 1303: "Natural Resources Management", 
    1401: "Architecture", 
    1501: "Area, Ethnic, and Civilization Studies", 
    1901: "Communications", 1902: "Journalism", 1903: "Mass Media", 1904: "Advertising and Public Relations", 
    2001: "Communication Technologies", 
    2100: "Computer Science and IT", 2101: "Computer Programming", 2102: "Computer Science", 2105: "Information Sciences", 
    2106: "Information Security", 2107: "Computer Networking", 
    2201: "Cosmetology and Culinary Arts", 
    2300: "Education", 2301: "Educational Administration", 2303: "School Counseling", 2304: "Elementary Education", 
    2305: "Mathematics Education", 2306: "Physical and Health Education", 2307: "Early Childhood Education", 
    2308: "Science Education", 2309: "Secondary Education", 2310: "Special Education", 2311: "Social Science Education", 
    2312: "Teacher Education", 2313: "Language and Drama Education", 2314: "Art and Music Education", 2399: "Miscellaneous Education", 
    2400: "Engineering", 2401: "Aerospace Engineering", 2402: "Biological Engineering", 2403: "Architectural Engineering", 
    2404: "Biomedical Engineering", 2405: "Chemical Engineering", 2406: "Civil Engineering", 2407: "Computer Engineering", 
    2408: "Electrical Engineering", 2409: "Environmental Engineering", 2410: "Industrial Engineering", 2411: "Materials Engineering", 
    2412: "Mechanical Engineering", 2413: "Metallurgical Engineering", 2414: "Mining Engineering", 2415: "Naval Architecture", 
    2416: "Nuclear Engineering", 2417: "Petroleum Engineering", 2418: "Miscellaneous Engineering", 2419: "Robotics Engineering", 
    2499: "Engineering Technologies", 2500: "Engineering Tech: General", 2501: "Drafting and Design", 2502: "Electrical Engineering Tech", 
    2503: "Industrial Production Tech", 2504: "Mechanical Engineering Tech", 2599: "Miscellaneous Engineering Tech", 
    2601: "Biology", 2602: "Biochemistry", 2603: "Botany", 
    2901: "Mathematics", 3000: "Statistics", 3301: "Social Work", 3302: "Public Administration", 
    3401: "Physical Sciences", 3402: "Astronomy", 3600: "Chemistry", 3601: "Geology", 3602: "Geosciences", 3603: "Oceanography", 
    3604: "Physics", 3605: "Meteorology", 3606: "Planetary Science", 3607: "Materials Science", 3608: "Atmospheric Science", 
    3609: "Environmental Science", 3611: "Space Science", 3699: "Misc. Physical Sciences", 
    3700: "Social Sciences", 3701: "Anthropology", 3702: "Archaeology", 
    3801: "Philosophy and Religion", 
    4000: "Economics", 4001: "Sociology", 4002: "Political Science", 4005: "Geography", 4006: "Criminal Justice", 
    4007: "Public Policy", 4801: "Law and Legal Studies", 
    5000: "Liberal Arts", 5001: "English Literature", 5002: "Linguistics", 5003: "Foreign Languages", 5004: "Comparative Literature", 
    5005: "Philosophy", 5006: "Religious Studies", 5007: "Humanities", 5008: "Interdisciplinary Humanities", 5098: "Miscellaneous Humanities", 
    5102: "Music", 5200: "Fine Arts", 5201: "Design and Applied Arts", 5202: "Graphic Design", 5203: "Photography", 5205: "Film", 5206: "Theater Arts", 
    5299: "Miscellaneous Fine Arts", 
    5401: "Business", 5402: "Accounting", 5403: "Finance", 5404: "Human Resources", 
    5500: "Management Information Systems", 5501: "Marketing", 5502: "Management Science", 5503: "Supply Chain Management", 
    5504: "Real Estate", 5505: "Entrepreneurship", 5506: "International Business", 5507: "Business Administration", 5599: "Miscellaneous Business", 
    5601: "Consumer Sciences", 5701: "Public Health", 
    6000: "Nursing", 6001: "Pharmacy", 6002: "Health Administration", 6003: "Public Health", 6004: "Medical Assisting", 
    6005: "Clinical Sciences", 6006: "Veterinary Sciences", 6007: "Health Services", 6099: "Miscellaneous Health Professions", 
    6100: "General Medicine", 6102: "Dentistry", 6103: "Pharmacy Technician", 6104: "Radiological Sciences", 6199: "Miscellaneous Medical Sciences", 
    6105: "Biomedical Sciences", 6106: "Nutritional Science", 6107: "Athletic Training", 6108: "Therapeutic Sciences", 
    6109: "Mental Health Services", 6110: "Community Health Services", 
    6200: "Psychology", 6201: "Clinical Psychology", 6202: "Industrial Psychology", 6203: "Developmental Psychology", 
    6204: "Cognitive Psychology", 6205: "Forensic Psychology", 6206: "Organizational Psychology", 6207: "Educational Psychology", 
    6209: "Experimental Psychology", 6210: "Social Psychology", 6211: "Counseling Psychology", 6212: "Abnormal Psychology", 6299: "Miscellaneous Psychology", 
    6402: "Human Development", 
    6403: "Social Work", 3202: "Military Science and Leadership", 4901: "Mechanical Engineering", 4101: "General Engineering", 
    5901: "Educational Leadership", 5301: "Business Administration", 3501: "Health Professions"
}
census_df['DEGFIELDD_NAME'] = census_df['DEGFIELDD'].map(degree_field_map)

In [ ]:
# check missing  values
census_df.isna().sum()

In [ ]:
#Drop missing valeus
census_df=census_df.dropna(how='any')

In [ ]:
# rename the occupation code
census_df=census_df.rename(columns={'OCCSOC':'OCC_CODE'})

In [ ]:
#check data structure
census_df.head()

In [ ]:
#save cleand census data 
census_df.to_csv('./Data/census_data.csv',index=False)

## 3. Data Merging :Combining Census and BLS Data

In [ ]:

# Rename columns in the BLS DataFrame for clarity
bls_clean.rename(columns={'AREA_TITLE': 'STATE'}, inplace=True)

In [ ]:
# Merge the DataFrames on OCC_CODE and STATE 
merged_df = pd.merge(census_df, bls_clean, on=['OCC_CODE', 'STATE'], how='inner')

In [ ]:
# check if is there any missing values before procedding 
merged_df.isna().sum()

In [ ]:
#check the data Structure 
merged_df.info()

In [ ]:
# display the data Statistic
merged_df.describe()

In [ ]:
merged_df.columns

In [ ]:
# Define the desired column order (update to match actual columns in merged_df)

# Get the actual columns present in merged_df

actual_columns = list(merged_df.columns)

# List of columns we want, in preferred order (will only keep those that exist)

desired_order = [
    'PRIM_STATE', 'STATE',   # Geographic Information
    'SEX', 'AGE', 'EDUCD', 'EDUCATION_LABEL', 'DEGFIELDD',  # Demographics and Education
    'OCC_CODE', 'OCC_TITLE',  # Occupation Details
    'INCTOT', 'TOT_EMP', 'LOC_QUOTIENT', 'JOBS_1000',  # Employment/Income Statistics
    'H_MEAN', 'A_MEAN'  # Wage Information
]

# Filter desired_order to only include columns that exist in merged_df

final_order = [col for col in desired_order if col in actual_columns]

if not final_order:

    raise ValueError('No matching columns found for reordering. Please check merged_df columns.')

# Reindex the DataFrame with the filtered order

merged_df = merged_df[final_order]


# Renaming columns for clarity (no more than two words)

rename_mapping = {
    'PRIM_STATE': 'State Abbreviation',
    'STATE': 'State',
    'SEX': 'Gender',
    'AGE': 'Age',
    'EDUCD': 'Education Code',
    'EDUCATION_LABEL': 'Education Level',
    'DEGFIELDD': 'Degree Field',
    'OCC_CODE': 'Occupation Code',
    'OCC_TITLE': 'Occupation',
    'INCTOT': 'Annual Income',
    'TOT_EMP': 'Employment',
    'LOC_QUOTIENT': 'Location Quotient',
    'JOBS_1000': 'Jobs per 1000',
    'H_MEAN': 'Hourly Mean',
    'A_MEAN': 'Annual Mean Wage'
}

# Only rename columns that exist in merged_df

rename_mapping = {k: v for k, v in rename_mapping.items() if k in merged_df.columns}

merged_df.rename(columns=rename_mapping, inplace=True)

# Display the updated DataFrame columns for verification

print('Final columns:', list(merged_df.columns))

In [ ]:
# Display the updated DataFrame columns as a list for better readability
print(list(merged_df.columns))

In [ ]:
merged_df.head()

In [ ]:
# Check for duplicate rows in the final dataset before saving
num_duplicates = merged_df.duplicated().sum()
print('Number of duplicate rows before saving:', num_duplicates)
# Remove duplicate rows from the final dataset
merged_df = merged_df.drop_duplicates()
print('Shape after removing duplicates:', merged_df.shape)

In [ ]:
# Save the cleaned data to a CSV file
merged_df.to_csv('./Data/cleaned_high_pay_data.csv', index=False)

## Final Data Validation and Summary
A quick check to confirm the final cleaned dataset is as expected.

In [ ]:
# Show shape and a sample of the cleaned, merged data
print('Final dataset shape:', merged_df.shape)
display(merged_df.head())
# Check for missing values in any column
print('Missing values per column:')
print(merged_df.isna().sum())